In [ ]:
import pandas as pd
import os

pd.options.display.float_format = '{:1.2f}'.format
pd.options.display.max_columns = None
pd.options.display.max_rows = None

In [ ]:
files = 0
for file in os.listdir("raw_sold"):
    files += 1
print(f"Number of sold files: {files}")

In [ ]:
df = pd.read_csv(f"raw_sold/CRMLSSold202502.csv", encoding='latin-1')

print('sold data:')
df.head()


In [ ]:
df.columns

In [ ]:
df.describe()

In [ ]:
merged_df = None
folder_path = "raw_sold"
for file in os.listdir(folder_path):
    df = pd.read_csv(f"{folder_path}/{file}", encoding='latin-1')
    Residential_df = df[df['PropertyType'] == 'Residential']
    if merged_df is None:
        merged_df = df
        merged_residential_df = Residential_df
    else:
        merged_df = pd.concat([merged_df, df])
        merged_residential_df = pd.concat([merged_residential_df, Residential_df])

In [ ]:
merged_df.describe()

In [ ]:
merged_residential_df.describe()

In [ ]:
print("rows in merged df: " + str(len(merged_df)))
print("rows in merged residential df: " + str(len(merged_residential_df)))
print("difference in rows: " + str(len(merged_df) - len(merged_residential_df)))

In [ ]:
# merged_residential_df.to_csv("sold_merged_residential.csv", index=False)

Merged analysis

In [ ]:
df = pd.read_csv("sold_merged_residential.csv", encoding='latin-1')
df['PropertyType'].unique()

In [ ]:
nulls = df.isnull().sum()

total_rows = len(df)

useless_Residential_columns = []

for column, null_count in nulls.items():
    percentage = (null_count / total_rows)
    if percentage > .9:
        print(f"{column}: {percentage:.2%} null")
        useless_Residential_columns.append(column)

the columns with > 90% nulls were put into a list called "useless_Residential_columns"

In [ ]:
Price_LivingArea_DOM_df = df[['ClosePrice', 'LivingArea', 'DaysOnMarket']]
Price_LivingArea_DOM_df.describe()

,ClosePrice,LivingArea,DaysOnMarket
count,397601.00,397374.00,397603.00
mean,1185616.36,1904.35,37.34
std,5922380.41,27017.81,53.54
min,0.00,0.00,-288.00
25%,575000.00,1247.00,8.00
50%,820000.00,1641.00,19.00
75%,1300000.00,2217.00,48.00
max,989500000.00,17021321.00,12430.00


In [ ]:
# Price_LivingArea_DOM_df.to_csv("Price_LivingArea_DOM_sold.csv", index=False)

FRED Analysis

In [ ]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates=['observation_date'])
mortgage = mortgage[mortgage['observation_date'] >= '2024-01-01'].reset_index().drop(columns=['index'])
mortgage.columns = ['date', 'rate_30yr_fixed']

In [ ]:
mortgage.head()

,date,rate_30yr_fixed
0,2024-01-04,6.62
1,2024-01-11,6.66
2,2024-01-18,6.60
3,2024-01-25,6.69
4,2024-02-01,6.63


In [ ]:
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = (
 mortgage.groupby('year_month')['rate_30yr_fixed']
 .mean()
 .reset_index()
)
mortgage_monthly.head()

,year_month,rate_30yr_fixed
0,2024-01,6.64
1,2024-02,6.78
2,2024-03,6.82
3,2024-04,6.99
4,2024-05,7.06


In [ ]:
sold = pd.read_csv('/week 1/sold_merged_residential.csv', encoding='latin-1')
sold['year_month'] = pd.to_datetime(sold['CloseDate']).dt.to_period('M')
sold[['CloseDate', 'year_month']].head()

In [ ]:
sold_with_rates = sold.merge(mortgage_monthly, on='year_month', how='left')
print(sold_with_rates['rate_30yr_fixed'].isnull().sum())

In [ ]:
sold_with_rates[
    ['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']
].head()

,CloseDate,year_month,ClosePrice,rate_30yr_fixed
0,2024-01-26,2024-01,240000.00,6.64
1,2024-01-05,2024-01,815000.00,6.64
2,2024-01-05,2024-01,810000.00,6.64
3,2024-01-30,2024-01,858000.00,6.64
4,2024-01-29,2024-01,1890500.00,6.64


In [ ]:
sold_with_rates[
    ['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']
].drop_duplicates('year_month').head(100)

,CloseDate,year_month,ClosePrice,rate_30yr_fixed
0,2024-01-26,2024-01,240000.00,6.64
11203,2024-02-07,2024-02,1249000.00,6.78
24266,2024-03-06,2024-03,1290000.00,6.82
40050,2024-04-02,2024-04,250000.00,6.99
57345,2024-05-31,2024-05,3300000.00,7.06
75819,2024-06-28,2024-06,4250000.00,6.92
92444,2024-07-31,2024-07,929000.00,6.85
110253,2024-08-09,2024-08,210000.00,6.50
127044,2024-09-12,2024-09,1090000.00,6.18
141599,2024-10-04,2024-10,208000.00,6.43


In [ ]:
# sold_with_rates.to_csv(str(parent_dir)+'/week 2-3/FRED_rates_sold.csv', index=False)

Data Cleaning